# **Geometric Parametrization**
This lab focuses on performing a ROM on a parametric system with variable domain geometry.

In [ ]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities
import time

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

In [ ]:
geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = 1.0e-6
geometry_utilities_config.tolerance2_d = 1.0e-12
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

## Poisson problem on variable geometry

Solve the following equation on the domain ${\tilde{\Omega}} = \tilde{\Omega}_1 \cup \tilde{\Omega}_2 =  (0, 1) \times (0, 1) \cup (1, 1+
\mu) \times (0, 1)$

$$
\begin{cases}
\nabla \cdot (\nabla u) = f & \text{in } \tilde{Ω}\\
u = 0 & \text{in } \partial \tilde{Ω}
\end{cases}
$$

The parametric space is $\mathcal P = [1, 3.5]$.

### Map to reference domain

**Goal**: build the ROM space using a reference domain

We choose as **reference domain** $\Omega$ the case $\mu = 1$.

Thus, $\Omega = \Omega_1 \cup \Omega_2 = [0,1] \times [0,1] \cup [1,2] \times [0,1]$.

The affine transformations are the following:
$$\tilde{\mathbf{x}} = \Phi_{\Omega_1}(\mathbf{x}, \mu) = \mathbb{I}\mathbf{x} + \mathbf{0}= \begin{bmatrix}
1 & 0 \\
0 & 1 
\end{bmatrix}\mathbf{x} + \begin{pmatrix}
0\\
0
\end{pmatrix} \quad \forall \tilde{\mathbf{x}} \in \tilde{\Omega_1}$$
$$\tilde{\mathbf{x}} = \Phi_{\Omega_2}(\mathbf{x}, \mu) = \mathbb{A}\mathbf{x} + \mathbf{c} = \begin{bmatrix}
\mu & 0 \\
0 & 1 
\end{bmatrix}\mathbf{x} + \begin{pmatrix}
1-\mu\\
0
\end{pmatrix} \quad \forall \tilde{\mathbf{x}} \in \tilde{\Omega_2}$$

Notice that
$$J_{\Phi_{\Omega_2}} =\begin{bmatrix}
\mu & 0 \\
0 & 1 
\end{bmatrix} ⇒ J^{-1}_{\Phi_{\Omega_2}} =\begin{bmatrix}
\frac{1}{\mu} & 0 \\
0 & 1 
\end{bmatrix}$$
and $|J_{\Phi_{\Omega_2}}| = \mu$

In [ ]:
geometric_tol = geometry_utilities_config.tolerance1_d 

In [ ]:
def Map(points, mu):
  numPoints = points.shape[1]
  mappedPoints = np.copy(points)

  for p in range(1, numPoints):
    if (points[0, p] > 1.0 + geometric_tol):
      mappedPoints[0, p] = mu * points[0, p] + (1. - mu)
  return mappedPoints

### Problem data

For this Lab we would like to find the solution
$$u = 16 x y (1 + \mu - x) (1-y)$$

Thus, the forcing term reads
$$f = 32 [x(1+\mu-x) + y(1-y)]$$

In [ ]:
def forcing_term(x, y, z):
    return 32.0 * (y * (1.0 - y) + x * (1.0 + MU_TILDE - x))
def exact_solution(x, y, z):
    return 16.0 * (y * (1.0 - y) * x * (1.0 + MU_TILDE - x))
def exact_derivative_solution(x, y, z):
    return np.array([\
        16.0 * (1.0 + MU_TILDE - 2.0 * x) * y * (1.0 - y),\
        16.0 * (1.0 - 2.0 * y) * x * (1.0 + MU_TILDE - x),\
        0.0])

### Forcing Terms with map applied

The problem on the reference domain $Ω$ shall be computing applying the transformation function to the original problem.

Thus we will have:

$$\tilde{a}(\tilde{u}, \tilde{v}) = \int_{\tilde{\Omega}} \tilde{\nabla} \tilde{u}(\tilde{\mathbf{x}})\cdot \tilde{\nabla} \tilde{v}(\tilde{\mathbf{x}}) = \int_{\Omega = \Phi^{-1}(\tilde{\Omega})} \tilde{\nabla} \tilde{u}(\Phi(\mathbf{x})) \cdot \tilde{\nabla} \tilde{v}(\Phi(\mathbf{x}))|J_{\Phi}| ⇒$$
$$a(u, v) = \int_{\Omega = \Phi^{-1}(\tilde{\Omega})} [J_{\Phi}^{-1}\nabla u(\mathbf{x})] \cdot [J_{\Phi}^{-1}\nabla v(\mathbf{x})]|J_{\Phi}|$$
and:
$$\tilde{f}(\tilde{v}) = \int_{\tilde{\Omega}} \tilde{f}(\tilde{\mathbf{x}})\tilde{v}(\tilde{\mathbf{x}}) = \int_{\Omega = \Phi^{-1}(\tilde{\Omega})} \tilde{f}(\Phi(\mathbf{x})) \tilde{v}(\Phi(\mathbf{x}))|J_{\Phi}| ⇒$$
$$f(v) = \int_{\Omega = \Phi^{-1}(\tilde{\Omega})} f(\mathbf{x}) v(\mathbf{x})|J_{\Phi}|$$

**NB** $\tilde{f}$ and $f$ are NOT the same function, as $f = \tilde{f} \circ \Phi$ !!!

In [ ]:
def forcing_term_ref_11(x, y, z):
    if (x > (1.0 + geometric_tol)):
        return 0.
    return 32.0 * (y * (1.0 - y) + x * (1.0 - x))
def forcing_term_ref_12(x, y, z):
    if (x > (1.0 + geometric_tol)):
        return 0.
    return 32.0 * x
def forcing_term_ref_21(x, y, z):
    if (x <= (1.0 + geometric_tol)):
        return 0.
    return 32.0 * (y * (1.0 - y))
def forcing_term_ref_22(x, y, z):
    if (x <= (1.0 + geometric_tol)):
        return 0.
    return 32.0 * (x * (2.0 - x))
def forcing_term_ref_23(x, y, z):
    if (x <= (1.0 + geometric_tol)):
        return 0.
    return 32.0 * (2.0 - x)

### Function for assembling the Offline problem

In [ ]:
def omega_tilde_1(x, y, z):
    if (x <= (1.0 + geometric_tol)):
        return 1.0
    return 0.0
def omega_tilde_2_1(x, y, z):
    A_array = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) # 3x3 matrix by column
    if (x > (1.0 + geometric_tol)):
        A_array[0] = 1.
    return A_array # diffusion matrix term A_matrix(i, j) = A_array(i + 3j), (i,j) \in [0,1,2]
def omega_tilde_2_2(x, y, z):
    A_array = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) # 3x3 matrix by column
    if (x > (1.0 + geometric_tol)):
        A_array[4] = 1.
    return A_array # diffusion matrix term A_matrix(i, j) = A_array(i + 3j), (i,j) \in [0,1,2]

**Let us code the OFFLINE PHASE**

In [ ]:
# Export folder
export_file_path = "./Export/Test_1"
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = export_file_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = export_file_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

In [ ]:
pde_domain = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D()
pde_domain.vertices = np.array([[0.0, 2.0, 2.0, 0.0],
                                [0.0, 0.0, 1.0, 1.0],
                                [0.0, 0.0, 0.0, 0.0]])
pde_domain.shape_type = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D.Domain_Shape_Types.parallelogram
pde_domain.area = 2.0

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D.triangular
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes.fem_pcc
method_order = 1
mesh_size = 0.00025

**CASE 1** - Creating Mesh Non-Conformed to the interface

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.create_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          pde_domain,
                                                          mesh_size,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

In [ ]:
vtk_utilities.export_mesh(export_mesh_path, mesh)
other_ut.plot_mesh(mesh)

**CASE 2** - Importing Mesh Conformed to the interface

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D.triangular_simple_importer
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes.fem_pcc
import_mesh_folder = "../Meshes/Mesh6"
method_order = 1

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.import_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          import_mesh_folder,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

In [ ]:
vtk_utilities.export_mesh(export_mesh_path, mesh)
other_ut.plot_mesh(mesh)

### FEM space (the High Fidelity approximation)

In [ ]:
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet.marker = 1

boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_dirichlet,
    3: info_dirichlet,
    4: info_dirichlet,
    5: info_dirichlet,
    6: info_dirichlet,
    7: info_dirichlet,
    8: info_dirichlet
}

In [ ]:
mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

trial_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)
test_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()

trial_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
trial_dofs_data = dof_manager.create_do_fs_2_d(trial_mesh_dofs_info, mesh_connectivity_data)
test_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
test_dofs_data = dof_manager.create_do_fs_2_d(test_mesh_dofs_info, mesh_connectivity_data)

In [ ]:
trial_n_dofs = trial_dofs_data.number_do_fs
trial_n_strongs = trial_dofs_data.number_strongs
test_n_dofs = test_dofs_data.number_do_fs
test_n_strongs = test_dofs_data.number_strongs

In [ ]:
print("trial dofs\t", "trial stgs\t", "test dofs\t", "test stgs\t")
print(trial_n_dofs,"\t", trial_n_strongs,"\t", test_n_dofs,"\t", test_n_strongs)

### Assemble linear system exploting affinity
We define everything that is parameter independent:
$$\mathbb{A}_i,\ i \in \{0,\dots, q_a\} \quad \mathbf{f}_j,\ j \in \{0,\dots, q_f\}.$$
Moreover, we define the matrix $\mathbb{X}$ related to the scalar product of the problem at hand.
Finally, we create the parameter dependent variable:
$$θ^a_i(\boldsymbol{\mu}),\ i \in \{0,\dots, q_a\} \quad θ^f_j(\boldsymbol{\mu}),\ j \in \{0,\dots, q_f\}$$


In [ ]:
forcing_term_11 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       forcing_term_ref_11)
forcing_term_12 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       forcing_term_ref_12)
forcing_term_21 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       forcing_term_ref_21)
forcing_term_22 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       forcing_term_ref_22)
forcing_term_23 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       forcing_term_ref_23)

In [ ]:
diffusion_1 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_diffusion_operator(geometry_utilities,
                                                                                        mesh,
                                                                                        mesh_geometric_data,
                                                                                        trial_dofs_data,
                                                                                        test_dofs_data,
                                                                                        trial_reference_element_data,
                                                                                        test_reference_element_data,
                                                                                        omega_tilde_1)
diffusion_2_1 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_anysotropic_diffusion_operator(geometry_utilities,
                                                                                                      mesh,
                                                                                                      mesh_geometric_data,
                                                                                                      trial_dofs_data,
                                                                                                      test_dofs_data,
                                                                                                      trial_reference_element_data,
                                                                                                      test_reference_element_data,
                                                                                                      omega_tilde_2_1)
diffusion_2_2 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_anysotropic_diffusion_operator(geometry_utilities,
                                                                                                      mesh,
                                                                                                      mesh_geometric_data,
                                                                                                      trial_dofs_data,
                                                                                                      test_dofs_data,
                                                                                                      trial_reference_element_data,
                                                                                                      test_reference_element_data,
                                                                                                      omega_tilde_2_2)


A_1 = other_ut.make_np_sparse(diffusion_1.operator_dofs)
A_2_1 = other_ut.make_np_sparse(diffusion_2_1.operator_dofs)
A_2_2 = other_ut.make_np_sparse(diffusion_2_2.operator_dofs)

X = A_1 + A_2_1 + A_2_2

In [ ]:
### define the problem
AQH = [A_1, A_2_1, A_2_2]
fQH = [forcing_term_11, forcing_term_12, forcing_term_21, forcing_term_22, forcing_term_23]

def thetaA(mu):
  return [1.0, 1.0 / mu[0], mu[0]]
def thetaF(mu):
  return [1.0, mu[0], mu[0], mu[0] * mu[0] * mu[0], mu[0] * mu[0] * (1.0 - mu[0])]

We will define some useful functions to perform computations:

In [ ]:
def normX(v, X):
	return np.sqrt(np.transpose(v) @ X @ v)

def ProjectSystem(AQH, fQH, B):
    AQN = []
    fQN = []
    for AH in AQH:
        AQN.append(np.copy(np.transpose(B) @ AH @ B))
    for fH in fQH:
        fQN.append(np.copy(np.transpose(B) @ fH))
    return [AQN, fQN]

def Solve_full_order(AQH, fQH, thetaA_mu, thetaF_mu):
    A = thetaA_mu[0] * AQH[0]
    f = thetaF_mu[0] * fQH[0]
    for i in range(1, len(AQH)):
        A += thetaA_mu[i] * AQH[i]
    for i in range(1, len(fQH)):
        f += thetaF_mu[i] * fQH[i]
    return scipy.sparse.linalg.spsolve(A, f)

def Solve_reduced_order(AQN, fQN, thetaA_mu, thetaF_mu):
    A = thetaA_mu[0] * AQN[0]
    f = thetaF_mu[0] * fQN[0]
    for i in range(1, len(AQN)):
        A += thetaA_mu[i] * AQN[i]
    for i in range(1, len(fQN)):
        f += thetaF_mu[i] * fQN[i]
    return np.linalg.solve(A, f)

We here define the finite parametric space $\mathcal P_{train}$, with random uniform distributed realization of $\boldsymbol \mu$.
The cardinality of $\mathcal P_{train}$ is set to $M=100$.

In [ ]:
### define the training set
M = 100
mu1_range = [1.0, 3.5]
P = np.array([mu1_range])

training_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(M, P.shape[0]))

### POD



In [ ]:
def POD(AQH, fQH, X, N_max, tol):
    #### snapshot matrix creation
    snapshot_matrix = []

    for mu in training_set:
        snapshot = Solve_full_order(AQH, fQH, thetaA(mu), thetaF(mu))
        snapshot_matrix.append(np.copy(snapshot))
    
    snapshot_matrix = np.array(snapshot_matrix) 

    ### covariance matrix
    C = snapshot_matrix @ X @ np.transpose(snapshot_matrix) ## metti inner product
    L_e, VM_e = np.linalg.eig(C)
    eigenvalues = []
    eigenvectors = []

    for i in range(len(L_e)):
        eig_real = L_e[i].real
        eig_complex = L_e[i].imag
        assert np.isclose(eig_complex, 0.)
        eigenvalues.append(eig_real)
        eigenvectors.append(VM_e[i].real)

    total_energy = sum(eigenvalues)
    retained_energy_vector = np.cumsum(eigenvalues)
    relative_retained_energy = retained_energy_vector/total_energy

    if all(flag==False for flag in relative_retained_energy >= (1.0 - tol)):
        N = N_max
    else:
        N = np.argmax(relative_retained_energy >= (1.0 - tol)) + 1

    # Create the basis function matrix
    basis_functions = []
    for n in range(N):
        eigenvector =  eigenvectors[n]
        # basis = (1/np.sqrt(M))*np.transpose(snapshot_matrix)@eigenvector 
        basis = np.transpose(snapshot_matrix) @ eigenvector
        norm = normX(basis, X)
        # norm = np.sqrt(np.transpose(basis)@basis)
        basis /= norm
        basis_functions.append(np.copy(basis))

    return [N, np.transpose(np.array(basis_functions))]

### Offline Phase


In [ ]:
tol = 1.0e-7
N_max = 20

We perform now the POD as for comparison:

In [ ]:
### Compute POD
[N_POD, B_POD] = POD(AQH, fQH, X, N_max, tol)
[AQN_POD, fQN_POD] = ProjectSystem(AQH, fQH, B_POD)

### Online Phase

In the _online phase_ we can use all the pre-assembled quantities to generate a new solution for a new parameter. 



In [ ]:
mu_test = 3.5
MU_TILDE = mu_test

In [ ]:
def strong_solution_function(marker, x, y, z):  
    return exact_solution(x, y, z)

In [ ]:
def TestSingleParameter(AQH, fQH, AQN, fQN, B, mu):
    reduced_solution = Solve_reduced_order(AQN, fQN, thetaA(mu), thetaF(mu))
    full_solution = Solve_full_order(AQH, fQH, thetaA(mu), thetaF(mu))

    ###### plot #######
    proj_reduced_solution = B @ reduced_solution

    ### computing error
    error_function = full_solution - proj_reduced_solution
    error_norm_squared_component = np.transpose(error_function) @ X @ error_function
    abs_err_ROM = np.sqrt(abs(error_norm_squared_component))

    full_solution_norm_squared_component = np.transpose(full_solution) @ X @ full_solution
    rel_err_ROM = abs_err_ROM / np.sqrt(abs(full_solution_norm_squared_component))

    solution_strong = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function)

    map_mu = mu[0]
    mapped_mesh_coordinates = Map(mesh.cell0_ds_coordinates(), map_mu)
    u_h_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                                trial_dofs_data,
                                                                                                full_solution,
                                                                                                solution_strong)
    u_N_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                                trial_dofs_data,
                                                                                                proj_reduced_solution,
                                                                                                solution_strong)
    u_exact_on_dofs = polydim.pde_tools.assembler_utilities.pcc_2_d.evaluate_function_on_dofs(geometry_utilities,
                                                                                              mesh,
                                                                                              mesh_geometric_data,
                                                                                              trial_dofs_data,
                                                                                              trial_reference_element_data,
                                                                                              exact_solution)
    u_ex_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          u_exact_on_dofs.function_dofs,
                                                                                          u_exact_on_dofs.function_strong)

    exact_solution_norm_squared_component = np.transpose(u_exact_on_dofs.function_dofs) @ X @ u_exact_on_dofs.function_dofs

    other_ut.plot_solution_on_coordinates(mapped_mesh_coordinates, u_h_on_cell0Ds.numeric_solution, "FULL Solution")
    other_ut.plot_solution_on_coordinates(mapped_mesh_coordinates, u_N_on_cell0Ds.numeric_solution, "REDC Solution")
    other_ut.plot_solution_on_coordinates(mapped_mesh_coordinates, u_ex_on_cell0Ds.numeric_solution, "EXCT Solution")

    error_L2 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_l2(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                          full_solution,
                                                                          solution_strong,
                                                                       exact_solution)
    error_H1 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_h1(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                          full_solution,
                                                                          solution_strong,
                                                                       exact_derivative_solution)

    abs_err_L2 = error_L2.error_l2
    abs_err_H1 = error_H1.error_h1
    rel_err_L2 = error_L2.error_l2 / np.sqrt(abs(exact_solution_norm_squared_component))
    rel_err_H1 = error_H1.error_h1 / np.sqrt(abs(exact_solution_norm_squared_component))
  
    return [rel_err_ROM, abs_err_ROM, rel_err_L2, abs_err_L2, rel_err_H1, abs_err_H1]

In [ ]:
[rel_err_ROM, abs_err_ROM, rel_err_L2, abs_err_L2, rel_err_H1, abs_err_H1] = TestSingleParameter(AQH, fQH, AQN_POD, fQN_POD, B_POD, [mu_test])

print("DOFs","N","rel_err_ROM","abs_err_ROM","rel_err_L2","abs_err_L2","rel_err_H1","abs_err_H1")
print(trial_n_dofs,\
          N_POD,\
          '{:.4e}'.format(np.mean(rel_err_ROM)),\
          '{:.4e}'.format(np.mean(abs_err_ROM)),\
          '{:.4e}'.format(np.mean(rel_err_L2)),\
          '{:.4e}'.format(np.mean(abs_err_L2)),\
          '{:.4e}'.format(np.mean(rel_err_H1)),\
          '{:.4e}'.format(np.mean(abs_err_H1)))

We can now compute an error analysis over the parametric space, together with a _speed-up_ anaslysis.

The speed-up is an index that evaluated how many ROM solution I can obtain in the time of a FOM simulation.

In [ ]:
def Avg_error(AQH, fQH, AQN, fQN, B):
    ### compute avg error
    abs_err = []
    rel_err = []
    testing_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(100, P.shape[0]))
    speed_up = []

    print("Computing error and speedup analysis...")

    for mu in testing_set:
        ##### full #####
        start_fom = time.time()
        full_solution = Solve_full_order(AQH, fQH, thetaA(mu), thetaF(mu))
        time_fom = time.time() - start_fom

        #### reduced #####
        start_rom = time.time()
        reduced_solution = Solve_reduced_order(AQN, fQN, thetaA(mu), thetaF(mu))
        time_rom = time.time() - start_rom

        speed_up.append(time_fom / time_rom)

        proj_reduced_solution = B @ reduced_solution

        ### computing error
        error_function = full_solution - proj_reduced_solution
        error_norm_squared_component = np.transpose(error_function) @ X @ error_function
        absolute_error = np.sqrt(abs(error_norm_squared_component))
        abs_err.append(absolute_error)

        full_solution_norm_squared_component = np.transpose(full_solution) @  X @ full_solution
        relative_error = absolute_error/np.sqrt(abs(full_solution_norm_squared_component))
        rel_err.append(relative_error)
    
    return [rel_err, abs_err, speed_up]

In [ ]:
[rel_err_POD, abs_err_POD, speed_up_POD] = Avg_error(AQH, fQH, AQN_POD, fQN_POD, B_POD)
print("- Average POD relative error = ", '{:.16e}'.format(np.mean(rel_err_POD)) )
print("- Average POD absolute error = ", '{:.16e}'.format(np.mean(abs_err_POD)) )
print("- Average POD speed_up       = ", '{:.16e}'.format(np.mean(speed_up_POD)))